In [ ]:
import pandas as pd
import numpy as np

In [ ]:
bp_data_path = '/Users/jk1/Downloads/bp_timebin_6h/bp_timebins_6h.csv'

In [ ]:
bp_df = pd.read_csv(bp_data_path)

In [ ]:
bp_df.head()

In [ ]:
bp_df

In [ ]:
systole_df = bp_df[bp_df['variable'] == 'systole']
diastole_df = bp_df[bp_df['variable'] == 'diastole']
mitteldruck_df = bp_df[bp_df['variable'] == 'mitteldruck']

systole_df = systole_df.rename(columns={'Value': 'systole'})
diastole_df = diastole_df.rename(columns={'Value': 'diastole'})
mitteldruck_df = mitteldruck_df.rename(columns={'Value': 'mitteldruck'})

In [ ]:
bp_df = bp_df.merge(systole_df[['PatientID', 'datetime', 'systole']], on=['PatientID', 'datetime'], how='left')
bp_df = bp_df.merge(diastole_df[['PatientID', 'datetime', 'diastole']], on=['PatientID', 'datetime'], how='left')
bp_df = bp_df.merge(mitteldruck_df[['PatientID', 'datetime', 'mitteldruck']], on=['PatientID', 'datetime'], how='left')

In [ ]:
bp_df = bp_df.drop(columns=['variable', 'Value', 'VariableID', 'status'])

In [ ]:
bp_df = bp_df.drop_duplicates()

In [ ]:
def normalisation(x, prior_median):
    epsilon = 1
    # avoid division by zero by adding epsilon
    # as epsilon is 1 no great shift in values
    return (x + epsilon) / (prior_median + epsilon)

In [ ]:
bp_df

In [ ]:
patient_bp_df = bp_df[bp_df['PatientID'] == 23860]
bp_metrics=['systole', 'diastole', 'mitteldruck']

In [ ]:
for index, row in patient_bp_df.iterrows():
    time_of_bd = row['datetime']
    prior_bp_df = patient_bp_df[
        patient_bp_df['datetime'] < time_of_bd]

    prior_medians = {}
    for bp_metric in bp_metrics:
        prior_medians[bp_metric] = prior_bp_df[f'{bp_metric}'].median()

        if (np.isnan(prior_medians[bp_metric])) & (not np.isnan(row[bp_metric])):
            patient_bp_df.loc[(patient_bp_df['datetime'] == time_of_bd), f'{bp_metric}_normalised'] = 1
        else:
            patient_bp_df.loc[(patient_bp_df['datetime'] == time_of_bd), f'{bp_metric}_normalised'] = normalisation(
                row[bp_metric], prior_medians[bp_metric])



In [ ]:
patient_bp_df[['datetime', 'systole', 'systole_normalised', 'diastole', 'diastole_normalised', 'mitteldruck', 'mitteldruck_normalised']]